# AlphaPulse — Week 2: Quantitative Risk & Monte Carlo Simulation

**Project:** Infotact Solutions Project 2 (Portfolio Risk & Volatility Monitor)  
**Core Methods:** Daily Log Returns, 30-Day Rolling Volatility, Vectorized Covariance (`w^T Σ w`), 10,000+ Cholesky Monte Carlo Paths, 95% VaR/CVaR, and Sharpe Ratio

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from src.config import PORTFOLIO_ASSETS, get_settings
from src.quant_engine import (
    build_price_matrix,
    compute_log_returns,
    compute_portfolio_kpis,
    compute_rolling_volatility,
    export_tableau_csvs,
    load_market_prices,
    push_quant_to_huggingface,
    run_monte_carlo,
    save_quant_to_mongodb,
)

settings = get_settings()
raw_df = load_market_prices()
price_matrix = build_price_matrix(raw_df)
log_returns = compute_log_returns(price_matrix)
rolling_vol = compute_rolling_volatility(log_returns, window=30)
corr_matrix = log_returns.corr()

weights = np.array([a.weight for a in PORTFOLIO_ASSETS if a.asset_id in price_matrix.columns])
print(f"Price matrix: {price_matrix.shape} | Log returns: {log_returns.shape}")

## 1. Portfolio KPIs: Annualized Return, Volatility, Sharpe & 95% VaR/CVaR

In [ ]:
kpis = compute_portfolio_kpis(log_returns, weights, confidence=settings.var_confidence)
for k, v in kpis.items():
    print(f"{k:25s}: {v}")

## 2. Correlation Heatmap & 30-Day Rolling Volatility

In [ ]:
fig_corr = px.imshow(
    corr_matrix.round(2),
    text_auto=True,
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Cross-Asset Return Correlation Matrix (INR)",
)
fig_corr.show()

fig_vol = px.line(
    rolling_vol,
    title="30-Day Annualized Rolling Volatility",
    labels={"value": "Annualized Volatility", "date": "Date", "variable": "Asset"},
)
fig_vol.show()

## 3. 10,000-Path Cholesky Monte Carlo Simulation & Statistical Sanity Audit

In [ ]:
mc_result = run_monte_carlo(
    log_returns,
    weights,
    n_simulations=settings.mc_simulations,
    horizon_days=252,
    initial_capital=1_000_000.0,
    confidence=settings.var_confidence,
)

sim_mean_ret = float(mc_result["terminal_returns"].mean())
print(f"Historical Annualized Return : {kpis['annualized_return']:.2%}")
print(f"Simulated Mean 1Y Return     : {sim_mean_ret:.2%}")
print(f"1Y 95% Monte Carlo VaR (INR) : ₹{mc_result['mc_var_95_horizon_inr']:,.2f}")

fan = mc_result["percentile_fan"]
fig_fan = go.Figure()
fig_fan.add_trace(go.Scatter(x=fan["day"], y=fan["p95"], name="95th Pct", line={"dash": "dot"}))
fig_fan.add_trace(go.Scatter(x=fan["day"], y=fan["p50"], name="Median (P50)", line={"width": 3}))
fig_fan.add_trace(go.Scatter(x=fan["day"], y=fan["p05"], name="5th Pct (95% VaR)", line={"dash": "dot"}))
fig_fan.update_layout(title="Monte Carlo 1-Year Portfolio Fan (10,000 Paths, ₹10L Initial)")
fig_fan.show()

## 4. Export Tableau CSVs, Save to MongoDB Compass & Push to Hugging Face

In [ ]:
export_tableau_csvs(price_matrix, log_returns, rolling_vol, corr_matrix, mc_result)
save_quant_to_mongodb(kpis, corr_matrix, mc_result, settings)
push_quant_to_huggingface(settings)